
## Liquid Clustering

- [Liquid Clustering](https://delta.io/blog/liquid-clustering/)
  - [Delta: Optimize](https://delta.io/blog/delta-lake-optimize/)
  - [Delta: Transaction Log](https://www.databricks.com/blog/2019/08/21/diving-into-delta-lake-unpacking-the-transaction-log.html)
- [Databricks Datasets](https://docs.databricks.com/aws/en/discover/databricks-datasets): `/databricks-datasets`


## Data

We are going to use the `/databricks-datasets/asa/airlines/` data.

> 22 CSVs with data from `1987` until `2008`

In [ ]:
# display(dbutils.fs.ls('/databricks-datasets/asa/airlines/'))

In [ ]:
airlines = (
    spark.read
        .option("inferSchema", "true")
        .option("header", "true")
        .csv("/databricks-datasets/asa/airlines/")
)
airlines.count() # 123,534,969

## Writing

In [ ]:
%sql
create schema if not exists anthony_caliani;
drop table if exists anthony_caliani.airlines_delta;

#### No Clustering

In [ ]:
import re
from pyspark.sql import functions as f


def camel_to_snake(name):
    s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1).lower()


attributes = ["flight_id"] + [f.col(col).alias(camel_to_snake(col)) for col in airlines.columns]

(airlines
    .withColumn("flight_id", f.expr("uuid()"))
    .select(*attributes)
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("anthony_caliani.airlines_delta"))

In [ ]:
%sql  select * from anthony_caliani.airlines_delta limit 5;

In [ ]:
%sql 
-- num files.....: 99
-- size in bytes.: 5,944,713,305 (5.54 GB)
-- size / files..: 57 MB
describe detail anthony_caliani.airlines_delta;

#### Log History

In [ ]:
%sql 
describe history anthony_caliani.airlines_delta;

---

#### Cluster by Year

In [ ]:
%sql 
alter table anthony_caliani.airlines_delta cluster by (year);
optimize anthony_caliani.airlines_delta;

In [ ]:
%sql 
-- num files.....: 97
-- size in bytes.: 5,944,661,126 (5.54 GB)
-- size / files..: 58 MB
describe detail anthony_caliani.airlines_delta

In [ ]:
%sql 
describe history anthony_caliani.airlines_delta;

---

#### Cluster by Flight ID, Year

**[How do I trigger clustering?](https://delta.io/blog/liquid-clustering/)**  
You can also **manually trigger** a liquid clustering operation using the `OPTIMIZE` command.

In [ ]:
%sql 
alter table anthony_caliani.airlines_delta cluster by (flight_id, year);
optimize anthony_caliani.airlines_delta;

In [ ]:
%sql 
-- num files.....: 101
-- size in bytes.: 6,715,646,355 (6.25 GB)
-- size / files..: 63 MB
describe detail anthony_caliani.airlines_delta;